# Extension – WSD with biomedical BERT embeddings

In Tasks 2–6 all the methods choose among the senses of **WordNet**, a general dictionary that does not contain most medical meanings. Here we try a different approach that does not use WordNet at all.

We use a BERT model trained on biomedical texts to turn a text into a **vector** (embedding), so that texts with a similar meaning get similar vectors:
1. for each sense (CUI) we embed its **UMLS names and definition**;
2. for each abstract we embed the **context window** around the ambiguous word;
3. we choose the sense whose vector is **closest** to the context (cosine similarity).

It is the same idea as Lesk (compare the context with a description of each sense), but with vectors instead of shared words, and with the UMLS senses instead of WordNet. So no CUI → WordNet mapping is needed, and every sense can be found.

In [2]:
%pip install transformers torch
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
sys.path.append("..")                          # wsd_utils.py is in scripts/

import torch
import transformers
from transformers import AutoTokenizer, AutoModel
from wsd_utils import sample_instances, context_window

import json
import re
from nltk.corpus import stopwords
from wsd_utils import DATA_DIR, CONCEPT_MAP

STOP = set(stopwords.words("english"))


print("torch       ", torch.__version__)
print("transformers", transformers.__version__)

torch        2.14.0+cpu
transformers 5.17.0


**The model**

We use **SapBERT** (`cambridgeltl/SapBERT-from-PubMedBERT-fulltext`), a BERT model pre-trained on PubMed abstracts and then trained on UMLS so that different names of the same concept (e.g. *heart attack* and *myocardial infarction*) get very close vectors. As suggested by its authors, we take the vector of the special `[CLS]` token as the embedding of the whole text, and we normalize it to length 1, so the cosine similarity between two texts is just the dot product of their vectors.

In [4]:
MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()                                   # we only use the model, we do not train it


def embed(texts, batch_size=32):
    """One vector per text: the [CLS] vector of SapBERT, normalized to length 1."""
    vectors = []
    with torch.no_grad():                      # no training, so no gradients (faster, less memory)
        for i in range(0, len(texts), batch_size):
            batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                              max_length=128, return_tensors="pt")
            cls = model(**batch).last_hidden_state[:, 0]          # vector of the [CLS] token
            vectors.append(torch.nn.functional.normalize(cls, dim=1))
    return torch.cat(vectors)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
context = "The left ventricle of the heart pumps blood to the body"
senses = {
    "Heart Ventricle":     "heart ventricle: a chamber of the heart that pumps blood into the arteries",
    "Cerebral Ventricles": "cerebral ventricles: cavities in the brain filled with cerebrospinal fluid",
}

vectors = embed([context] + list(senses.values()))
similarities = vectors[1:] @ vectors[0]        # cosine similarity of each sense with the context
for name, s in zip(senses, similarities):
    print(f"{name:20} {s:.3f}")

Heart Ventricle      0.716
Cerebral Ventricles  0.453


**A description for each sense**

For each CUI we build a short text from the UMLS data downloaded for the full dataset (`umls_cui_info.json`): its **shortest names** (the most common ones, e.g. *eyelash*, *eyelashes*) followed by its first **English definition**. We use only English definitions, because the abstracts are in English. This text plays the role of the WordNet gloss in Lesk.

In [6]:

umls = json.loads((DATA_DIR.parent / "umls_cui_info.json").read_text(encoding="utf-8"))


def is_english(text):
    """True if at least 20% of the words are common English words (the, of, which...)."""
    words = re.findall(r"\w+", text.lower())
    return bool(words) and sum(w in STOP for w in words) / len(words) >= 0.2


def sense_text(cui, n_names=5):
    """Description of a sense: its shortest UMLS names and its first English definition."""
    info = umls.get(cui, {"names": [], "definitions": []})
    names = sorted(info["names"], key=len)[:n_names]
    english = [d for d in info["definitions"] if is_english(d)]
    return "; ".join(names) + (". " + english[0] if english else "")


for word in ["Cilia", "ALS"]:
    for label, cui in CONCEPT_MAP[word].items():
        print(f"{word:6} {label} {cui}  ->  {sense_text(cui)}")

Cilia  M1 C0008778  ->  cilia; cilium; cilium, nos; eukaryotic flagellum; cilium (cell structure). populations of thin, motile processes found covering the surface of ciliates (ciliophora) or the free surface of the cells making up ciliated epithelium. each cilium arises from a basic granule in the superficial layer of cytoplasm. the movement of cilia propels ciliates through the liquid in which they live. the movement of cilia on a ciliated epithelium serves to propel a surface layer of mucus or fluid. (king & stansfield, a dictionary of genetics, 4th ed)
Cilia  M2 C0015422  ->  cilia; cilias; eyelash; eyelashes; eyelashes set. anyone of the short hairs that grow on the edge of the eyelid.
ALS    M1 C0003372  ->  antilymphocyte serum; antilymphocyte serums; serum, antilymphocyte; serums, antilymphocyte; antilymphocyte globulin. serum containing gamma-globulins which are antibodies for lymphocyte antigens. it is used both as a test for histocompatibility and therapeutically in transpla

**Sense vectors**

We embed the description of every CUI of the dataset only once, and keep the vectors in a dictionary `CUI -> vector`. A few CUIs (retired from UMLS) have no names and no definition, so their description is empty and they can hardly be chosen.

In [7]:
ALL_CUIS = sorted({cui for senses in CONCEPT_MAP.values() for cui in senses.values()})
texts = [sense_text(cui) for cui in ALL_CUIS]
SENSE_VECTORS = dict(zip(ALL_CUIS, embed(texts)))

print(len(SENSE_VECTORS), "senses embedded,", sum(1 for t in texts if not t), "with an empty description")

410 senses embedded, 4 with an empty description


**Disambiguation**

For each abstract we embed the context window around the ambiguous word (the same window of 10 words as in Tasks 4 and 6) and choose the CUI of the word whose vector is closest to it.

In [8]:
def bert_wsd(contexts, word):
    """For each context, the CUI of the word whose description is closest (cosine similarity)."""
    cuis = list(CONCEPT_MAP[word].values())
    sense_matrix = torch.stack([SENSE_VECTORS[c] for c in cuis])      # one row per sense
    context_vectors = embed(contexts)                                 # one row per context
    similarities = context_vectors @ sense_matrix.T                   # contexts x senses
    return [cuis[i] for i in similarities.argmax(dim=1).tolist()]


instances = sample_instances("Cilia")
contexts = [context_window(inst["tagged"]) for inst in instances]
answers = bert_wsd(contexts, "Cilia")
correct = sum(a == inst["cui"] for a, inst in zip(answers, instances))
print(f"Cilia: {correct}/{len(instances)} correct")

Cilia: 14/20 correct


**Evaluation on the whole dataset**

We run the method on all 200 words, with the same sample of abstracts as Tasks 4 and 6 (10 per sense, same seed) and the same context window. The answers are saved in `bert_predictions.json`, so the notebook can be re-run without computing them again. Since the method answers directly with a CUI, the answer is correct if it is equal to the CUI of the abstract: no mapping is needed.

In [10]:
import time
from pathlib import Path
from wsd_utils import ALL_ACRONYMS, ALL_TERMS

CACHE = Path("bert_predictions.json")
predictions = json.loads(CACHE.read_text()) if CACHE.exists() else {}

start = time.time()
for word in ALL_ACRONYMS + ALL_TERMS:
    if word in predictions:                         # already computed
        continue
    instances = sample_instances(word)
    contexts = [context_window(inst["tagged"]) for inst in instances]
    answers = bert_wsd(contexts, word)
    predictions[word] = [{"cui": inst["cui"], "answer": a} for inst, a in zip(instances, answers)]

CACHE.write_text(json.dumps(predictions))
print(len(predictions), "words,", sum(len(rows) for rows in predictions.values()), "abstracts,",
      f"{time.time() - start:.0f} s")

200 words, 4170 abstracts, 124 s


In [11]:
import pandas as pd
from wsd_utils import MAPPING, mean


def word_accuracy(rows):
    """Share of abstracts where the chosen CUI is the correct one."""
    return sum(r["answer"] == r["cui"] for r in rows) / len(rows)


per_word = {w: word_accuracy(rows) for w, rows in predictions.items()}
random_choice = {w: 1 / len(CONCEPT_MAP[w]) for w in predictions}      # expected accuracy of a random guess

table = []
for name, words in [("all 200 words", ALL_ACRONYMS + ALL_TERMS),
                    ("our 39 words", [w for w in ALL_ACRONYMS + ALL_TERMS if w in MAPPING])]:
    acr = [w for w in words if w in ALL_ACRONYMS]
    ter = [w for w in words if w in ALL_TERMS]
    table.append({"words": name,
                  "acronyms": mean(per_word[w] for w in acr), "acronyms random": mean(random_choice[w] for w in acr),
                  "terms": mean(per_word[w] for w in ter),    "terms random": mean(random_choice[w] for w in ter)})

results = pd.DataFrame(table).set_index("words").round(3)
results

,acronyms,acronyms random,terms,terms random
words,,,,
all 200 words,0.875,0.486,0.711,0.489
our 39 words,0.844,0.479,0.718,0.500


**Observations**
- SapBERT reaches **0.844 on acronyms and 0.718 on terms** on our 39 words, against 0.257 and 0.371 of the best pywsd method in Task 4 on the same abstracts. The main reason is that it chooses directly among the UMLS senses, so every abstract can be disambiguated, while with WordNet more than half of them was impossible.
- It is far above a random choice (about 0.49), so it really uses the context.
- Unlike the WordNet methods, it works better on **acronyms** than on terms: the senses of an acronym are usually very different topics (e.g. *ALS*: a serum or a disease of motor neurons), while the senses of a term are often close (e.g. *milk* and *human milk*).
- The results on all 200 words are similar to the ones on our 39 words, so our selection was representative.

In [12]:
def sense_text(cui, n_names=5, use_names=True, use_definition=True):
    """Description of a sense: its shortest UMLS names and/or its first English definition."""
    info = umls.get(cui, {"names": [], "definitions": []})
    names = sorted(info["names"], key=len)[:n_names] if use_names else []
    english = [d for d in info["definitions"] if is_english(d)][:1] if use_definition else []
    return ". ".join(filter(None, ["; ".join(names)] + english))

**What matters? Context size and sense description**

We try a few variants on all the words: a larger context window (10, 25 and 50 words on each side; BERT is fast, so we are not limited by speed as with Wu-Palmer), and different sense descriptions (names only, definition only, both). We report all the variants, not only the best one.

In [13]:
def evaluate_variant(window, use_names=True, use_definition=True):
    """Average accuracy on acronyms and terms (all words) of one variant of the method."""
    texts = [sense_text(c, use_names=use_names, use_definition=use_definition) for c in ALL_CUIS]
    vectors = dict(zip(ALL_CUIS, embed(texts)))
    acc = {}
    for word in ALL_ACRONYMS + ALL_TERMS:
        instances = sample_instances(word)
        contexts = [context_window(inst["tagged"], n=window) for inst in instances]
        cuis = list(CONCEPT_MAP[word].values())
        similarities = embed(contexts) @ torch.stack([vectors[c] for c in cuis]).T
        answers = [cuis[i] for i in similarities.argmax(dim=1).tolist()]
        acc[word] = sum(a == inst["cui"] for a, inst in zip(answers, instances)) / len(instances)
    return mean(acc[w] for w in ALL_ACRONYMS), mean(acc[w] for w in ALL_TERMS)


VARIANTS = [
    ("window 10, names + definition", dict(window=10)),
    ("window 25, names + definition", dict(window=25)),
    ("window 50, names + definition", dict(window=50)),
    ("window 25, names only",         dict(window=25, use_definition=False)),
    ("window 25, definition only",    dict(window=25, use_names=False)),
]

table = []
for name, params in VARIANTS:
    t = time.time()
    acr, ter = evaluate_variant(**params)
    table.append({"variant": name, "acronyms": acr, "terms": ter})
    print(f"{name:32} {time.time() - t:5.0f} s")

variants = pd.DataFrame(table).set_index("variant").round(3)
variants

window 10, names + definition      165 s
window 25, names + definition     2829 s
window 50, names + definition      466 s
window 25, names only              262 s
window 25, definition only         295 s


,acronyms,terms
variant,,
"window 10, names + definition",0.875,0.711
"window 25, names + definition",0.841,0.719
"window 50, names + definition",0.839,0.715
"window 25, names only",0.806,0.678
"window 25, definition only",0.788,0.660


**Observations**
- A **larger context does not help**: acronyms go down from 0.875 (window 10) to 0.839 (window 50), terms stay about the same. SapBERT is trained on short texts (concept names), so its `[CLS]` vector works best with the few words closest to the ambiguous word; with a long context it gets diluted by other topics of the abstract (and very long windows are also cut at 128 tokens). For acronyms the closest words are often the decisive clue (e.g. the full name next to the abbreviation).
- **Names and definition together** give the best description: names only (0.806 / 0.678) and definition only (0.788 / 0.660) are both worse than the combination (0.841 / 0.719, same window).
- So the first version (window 10, names + definition), chosen before trying the variants, remains the best one.